In [ ]:
#nullable enable

public enum OrderStatus
{
    Created,
    InProgress,
    Completed,
    Cancelled
}

public delegate void OrderNotificationHandler(Order order, OrderStatus oldStatus, OrderStatus newStatus);

public abstract class Person
{
    public string FirstName { get; set; }
    public string LastName { get; set; }

    public Person(string firstName, string lastName)
    {
        FirstName = firstName;
        LastName = lastName;
    }

    public virtual string GetFullName() => $"{FirstName} {LastName}";
    public abstract string GetInfo();
}

public class Customer : Person
{
    public string PhoneNumber { get; set; }

    public Customer(string firstName, string lastName, string phoneNumber)
        : base(firstName, lastName)
    {
        PhoneNumber = phoneNumber;
    }

    public override string GetInfo() =>
        $"Заказчик: {GetFullName()} | Телефон: {PhoneNumber}";

    public void UpdatePhoneNumber(string newNumber)
    {
        PhoneNumber = newNumber;
    }

    public override string ToString() => GetInfo();
}

public class Employee : Person
{
    public string Position { get; set; }
    public int OrdersProcessed { get; private set; }

    public Employee(string firstName, string lastName, string position)
        : base(firstName, lastName)
    {
        Position = position;
        OrdersProcessed = 0;
    }

    public override string GetInfo() =>
        $"Сотрудник: {GetFullName()} | Должность: {Position} | Обработано заказов: {OrdersProcessed}";

    public void OnOrderStatusChanged(Order order, OrderStatus oldStatus, OrderStatus newStatus)
    {
        Console.WriteLine($"[Уведомление для {GetFullName()}]: Заказ #{order.OrderId} \"{order.Description}\" сменил статус: {oldStatus} -> {newStatus}");
    }

    public void IncrementOrdersProcessed()
    {
        OrdersProcessed++;
    }

    public void ResetOrdersProcessed()
    {
        OrdersProcessed = 0;
    }

    public override string ToString() => GetInfo();
}

public class Order
{
    private static int _nextId = 1;

    public int OrderId { get; }
    public string Description { get; set; }
    public Customer Customer { get; set; }
    public Employee? AssignedEmployee { get; private set; }
    public OrderStatus Status { get; private set; }
    public DateTime CreatedAt { get; }

    public event OrderNotificationHandler? StatusChanged;

    public Order(string description, Customer customer)
    {
        OrderId = _nextId++;
        Description = description;
        Customer = customer;
        Status = OrderStatus.Created;
        CreatedAt = DateTime.Now;
    }

    public void AssignEmployee(Employee employee)
    {
        if (AssignedEmployee != null)
        {
            StatusChanged -= AssignedEmployee.OnOrderStatusChanged;
        }

        AssignedEmployee = employee;
        StatusChanged += employee.OnOrderStatusChanged;
    }

    public void ChangeStatus(OrderStatus newStatus)
    {
        if (Status == newStatus) return;

        OrderStatus oldStatus = Status;
        Status = newStatus;

        StatusChanged?.Invoke(this, oldStatus, newStatus);

        if (newStatus == OrderStatus.Completed && AssignedEmployee != null)
        {
            AssignedEmployee.IncrementOrdersProcessed();
        }
    }

    public string GetOrderInfo() =>
        $"Заказ #{OrderId} [{Status}] от {CreatedAt:dd.MM.yyyy HH:mm} | \"{Description}\" | Клиент: {Customer.GetFullName()} | Ответственный: {(AssignedEmployee != null ? AssignedEmployee.GetFullName() : "Не назначен")}";

    public override string ToString() => GetOrderInfo();
}

public class Company
{
    public string Name { get; set; }
    public List<Employee> Employees { get; } = new();
    public List<Order> Orders { get; } = new();
    public List<Customer> Customers { get; } = new();

    public Company(string name)
    {
        Name = name;
    }

    public void AddEmployee(Employee employee)
    {
        Employees.Add(employee);
    }

    public void AddCustomer(Customer customer)
    {
        Customers.Add(customer);
    }

    public Order CreateOrder(string description, Customer customer)
    {
        var order = new Order(description, customer);
        Orders.Add(order);
        if (!Customers.Contains(customer))
        {
            Customers.Add(customer);
        }
        return order;
    }

    public void AssignEmployeeToOrder(int orderId, Employee employee)
    {
        var order = Orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order != null)
        {
            order.AssignEmployee(employee);
        }
    }

    public void ChangeOrderStatus(int orderId, OrderStatus newStatus)
    {
        var order = Orders.FirstOrDefault(o => o.OrderId == orderId);
        if (order != null)
        {
            order.ChangeStatus(newStatus);
        }
    }

    public List<Order> GetOrdersByStatus(OrderStatus status) =>
        Orders.Where(o => o.Status == status).ToList();

    public List<Order> FindOrdersByCustomer(Customer customer) =>
        Orders.Where(o => o.Customer == customer).ToList();

    public void PrintEmployeeReport()
    {
        Console.WriteLine($"Отчет по сотрудникам компании \"{Name}\":");
        foreach (var employee in Employees)
        {
            Console.WriteLine($"- {employee.GetFullName()} ({employee.Position}): обработано заказов — {employee.OrdersProcessed}");
        }
    }
}

Company company = new Company("ИТ-Сервис");

Employee emp1 = new Employee("Алексей", "Смирнов", "Старший менеджер");
Employee emp2 = new Employee("Ольга", "Иванова", "Технический специалист");
company.AddEmployee(emp1);
company.AddEmployee(emp2);

Customer cust1 = new Customer("Дмитрий", "Кузнецов", "+7 (999) 111-22-33");
Customer cust2 = new Customer("Елена", "Попова", "+7 (999) 444-55-66");
company.AddCustomer(cust1);
company.AddCustomer(cust2);

Console.WriteLine("=== ДОБАВЛЕНИЕ СУЩНОСТЕЙ В СИСТЕМУ ===");
Order order1 = company.CreateOrder("Разработка корпоративного сайта", cust1);
Order order2 = company.CreateOrder("Настройка сетевой инфраструктуры", cust2);
Order order3 = company.CreateOrder("Техническая поддержка серверов", cust1);

Console.WriteLine(order1);
Console.WriteLine(order2);
Console.WriteLine(order3);

Console.WriteLine("\n=== НАЗНАЧЕНИЕ ОТВЕТСТВЕННЫХ СОТРУДНИКОВ ===");
company.AssignEmployeeToOrder(order1.OrderId, emp1);
company.AssignEmployeeToOrder(order2.OrderId, emp2);
company.AssignEmployeeToOrder(order3.OrderId, emp1);

Console.WriteLine(order1);
Console.WriteLine(order2);
Console.WriteLine(order3);

Console.WriteLine("\n=== ИЗМЕНЕНИЕ СТАТУСОВ ЗАКАЗОВ (УВЕДОМЛЕНИЯ ЧЕРЕЗ ДЕЛЕГАТЫ И СОБЫТИЯ) ===");
company.ChangeOrderStatus(order1.OrderId, OrderStatus.InProgress);
company.ChangeOrderStatus(order2.OrderId, OrderStatus.InProgress);
company.ChangeOrderStatus(order1.OrderId, OrderStatus.Completed);
company.ChangeOrderStatus(order3.OrderId, OrderStatus.InProgress);
company.ChangeOrderStatus(order3.OrderId, OrderStatus.Completed);

Console.WriteLine("\n=== ПРОСМОТР ВСЕХ ЗАКАЗОВ ПО СТАТУСАМ ===");
Console.WriteLine("Заказы со статусом [Completed]:");
foreach (var order in company.GetOrdersByStatus(OrderStatus.Completed))
{
    Console.WriteLine($"  {order}");
}

Console.WriteLine("Заказы со статусом [InProgress]:");
foreach (var order in company.GetOrdersByStatus(OrderStatus.InProgress))
{
    Console.WriteLine($"  {order}");
}

Console.WriteLine("\n=== ПОИСК ЗАКАЗОВ ПО ЗАКАЗЧИКУ ===");
Console.WriteLine($"Заказы заказчика {cust1.GetFullName()}:");
foreach (var order in company.FindOrdersByCustomer(cust1))
{
    Console.WriteLine($"  {order}");
}

Console.WriteLine("\n=== ОТЧЕТ ПО СОТРУДНИКАМ ===");
company.PrintEmployeeReport();

=== ДОБАВЛЕНИЕ СУЩНОСТЕЙ В СИСТЕМУ ===
Заказ #1 [Created] от 19.11.2026 23:30 | "Разработка корпоративного сайта" | Клиент: Дмитрий Кузнецов | Ответственный: Не назначен
Заказ #2 [Created] от 19.11.2026 23:30 | "Настройка сетевой инфраструктуры" | Клиент: Елена Попова | Ответственный: Не назначен
Заказ #3 [Created] от 19.11.2026 23:30 | "Техническая поддержка серверов" | Клиент: Дмитрий Кузнецов | Ответственный: Не назначен

=== НАЗНАЧЕНИЕ ОТВЕТСТВЕННЫХ СОТРУДНИКОВ ===
Заказ #1 [Created] от 19.11.2026 23:30 | "Разработка корпоративного сайта" | Клиент: Дмитрий Кузнецов | Ответственный: Алексей Смирнов
Заказ #2 [Created] от 19.11.2026 23:30 | "Настройка сетевой инфраструктуры" | Клиент: Елена Попова | Ответственный: Ольга Иванова
Заказ #3 [Created] от 19.11.2026 23:30 | "Техническая поддержка серверов" | Клиент: Дмитрий Кузнецов | Ответственный: Алексей Смирнов

=== ИЗМЕНЕНИЕ СТАТУСОВ ЗАКАЗОВ (УВЕДОМЛЕНИЯ ЧЕРЕЗ ДЕЛЕГАТЫ И СОБЫТИЯ) ===
[Уведомление для Алексей Смирнов]: Заказ #1 "Разрабо